<a href="https://colab.research.google.com/github/xKDR/India-Built-and-Lit/blob/main/building_volume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building volume — reproducible pipeline

Computes annual **built-up volume** for every Indian district from the
Google Open Buildings 2.5D Temporal dataset, entirely inside Google Earth
Engine. Results come back with `getInfo()` — no Drive, no export tasks.

This notebook is **pure Python, single kernel** — run it top-to-bottom on
Google Colab (or locally). The district boundaries are fetched from GitHub,
so nothing else is needed.

**Output:** `bv_annual.csv` — one row per (district, year).


## 1 · Setup

In [1]:
# Colab ships earthengine-api; this just makes sure it's current.
!pip install -q -U earthengine-api

import ee

# Opens an auth pop-up on first run.
ee.Authenticate()

PROJECT = "gee-ntl-470405"          # <-- your GEE Cloud project id
ee.Initialize(project=PROJECT)
print("Earth Engine ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.6/479.6 kB 16.3 MB/s eta 0:00:00
Earth Engine ready.


## 2 · Parameters

`SCALE_M` is the **resampling knob**. Open Buildings is ~4 m native; reducing
at 100 m is far faster and, summed over a whole district, gives essentially
the same total — **but only for quantities that are linear in the pixel
values**, i.e. `band * pixel_area`. Earth Engine mean-pyramids continuous
bands before the reducer sees them, and mean-pyramiding preserves integrals.
A *threshold* does not commute with averaging, so `(band > t) * pixel_area`
computed at 100 m measures something else entirely — see section 4.

In [2]:
YEARS        = range(2016, 2024)      # Open Buildings 2.5D Temporal: 2016-2023
SCALE_M      = 100                    # reduction scale in metres (resampling knob)
BOUNDARY_URL = "https://raw.githubusercontent.com/xKDR/India-Built-and-Lit/main/data/boundaries/districts_simplified.geojson"
OUT_CSV      = "bv_annual.csv"
COLLECTION   = "GOOGLE/Research/open-buildings-temporal/v1"
ID_COLUMNS   = ["pc11_s_id", "pc11_d_id", "d_name"]

## 3 · Districts → `ee.FeatureCollection`

The simplified SHRUG district polygons are fetched from GitHub and loaded
inline — no GEE asset upload, no local file.


In [3]:
import json, urllib.request

with urllib.request.urlopen(BOUNDARY_URL) as resp:
    gj = json.load(resp)

feats = [
    ee.Feature(
        ee.Geometry(ft["geometry"], proj="EPSG:4326", geodesic=False),
        {k: ft["properties"].get(k) for k in ID_COLUMNS},
    )
    for ft in gj["features"]
]
districts = ee.FeatureCollection(feats)
print(districts.size().getInfo(), "districts loaded")

641 districts loaded


## 4 · Built-up volume and footprint per district per year

`volume_m3 = sum(building_height x pixel_area)` — linear in the pixel values,
so mean-pyramiding preserves it and the result is exact at any `SCALE_M`.

`footprint_m2 = sum(building_presence x pixel_area)` — the *expected* built
area. Also linear, so also exact at 100 m.

> **Why not `(building_height > 0) * pixel_area`?** That was the original
> definition here and it is wrong at any scale coarser than native. A 100 m
> cell holding one 100 m² building has pyramid-mean height ~0.05 m, passes
> `> 0`, and contributes the **full 10,000 m²**. It measures built-up
> *extent* — inflated by roughly 1/density — not footprint. Nationally it
> overstated footprint by ~14.5x and, since height = volume/footprint, it
> pushed mean building height down to an impossible ~0.6 m. It is retained
> below as `extent_m2` purely for comparison.

Thresholding presence (`presence >= 0.5`, the usual convention) would be
defensible, but only applied at native resolution — the same non-linearity
applies. Weighting by presence keeps the estimator linear, and avoids a cliff
that makes pixels near the cutoff flip in and out between years. The cost is
that presence confidence is uncalibrated, so the *level* is biased.

`reduceRegions` over all 641 districts at once overruns GEE's interactive
compute budget ("Computation timed out"). So we reduce in **chunks** and pull
each with `getInfo()`; `reduce_chunk` **auto-splits** any chunk that still
times out, down to single districts — heavy districts end up in tiny batches.

In [ ]:
import pandas as pd

# Rasters ship at 0.5 m in a per-tile UTM CRS. Only quantities LINEAR in the
# pixel values survive the reduction at SCALE_M -- see the note above.
NATIVE_PIXEL_AREA_M2 = 0.5 ** 2


def year_image(year):
    img = (ee.ImageCollection(COLLECTION)
           .filterDate(f"{year}-01-01", f"{year + 1}-01-01")
           .mosaic())
    px = ee.Image.pixelArea()
    h  = img.select("building_height")
    p  = img.select("building_presence")
    fc = img.select("building_fractional_count")

    volume    = h.multiply(px).rename("volume_m3")
    footprint = p.multiply(px).rename("footprint_m2")
    # building_fractional_count is a count PER NATIVE PIXEL, not a density, so
    # summing pyramid-averaged values silently drops the (SCALE_M/0.5)^2
    # aggregation factor. Convert to count per m2 first.
    count     = fc.divide(NATIVE_PIXEL_AREA_M2).multiply(px).rename("building_count")
    # The OLD footprint definition. Built-up extent at SCALE_M, NOT footprint.
    extent    = px.multiply(h.gt(0)).rename("extent_m2")

    return volume.addBands(footprint).addBands(count).addBands(extent)


BANDS = ["volume_m3", "footprint_m2", "building_count", "extent_m2"]


def reduce_chunk(img, chunk):
    """getInfo() a chunk of districts; on a compute timeout, split the chunk
    in half and recurse so heavy districts land in smaller batches."""
    try:
        fc = (img.reduceRegions(collection=ee.FeatureCollection(chunk),
                                reducer=ee.Reducer.sum(),
                                scale=SCALE_M, tileScale=16)
                 .select(ID_COLUMNS + BANDS, retainGeometry=False))
        return fc.getInfo()["features"]
    except ee.ee_exception.EEException as e:
        if len(chunk) == 1 or "timed out" not in str(e).lower():
            raise
        mid = len(chunk) // 2
        return reduce_chunk(img, chunk[:mid]) + reduce_chunk(img, chunk[mid:])


CHUNK = 50          # districts per getInfo() call; auto-splits on timeout

frames = []
for year in YEARS:
    img = year_image(year)
    recs = []
    for start in range(0, len(feats), CHUNK):
        recs += reduce_chunk(img, feats[start:start + CHUNK])
    part = pd.DataFrame([r["properties"] for r in recs])
    part["year"] = year
    frames.append(part)
    print(f"  {year}: {len(part)} districts")

## 5 · Assemble → `bv_annual.csv`

In [ ]:
df = pd.concat(frames, ignore_index=True)
df = df[df.footprint_m2 > 0].copy()
df["year"] = df["year"].astype(int)

# Mean building height over built area (m). Only meaningful because
# footprint_m2 now comes from the presence band -- against extent_m2 this
# lands near 0.6 m, which is the symptom of the bug described above.
df["mean_height_m"] = df.volume_m3 / df.footprint_m2
# Share of built-up extent that is actually building.
df["builtup_density"] = df.footprint_m2 / df.extent_m2.replace(0, pd.NA)

df = df[ID_COLUMNS + ["year"] + BANDS + ["mean_height_m", "builtup_density"]]
df = df.sort_values(["pc11_s_id", "pc11_d_id", "year"])
df.to_csv(OUT_CSV, index=False)
print(f"wrote {len(df)} rows -> {OUT_CSV}")
df.head()

## 6 · Plot — national built-up volume by year

In [ ]:
import matplotlib.pyplot as plt

national = df.groupby("year").volume_m3.sum()
ax = national.plot(marker="o", color="#f57d6a", figsize=(7, 4))
ax.set_ylabel("Total built-up volume (m^3)")
ax.set_title("India - total built-up volume by year")
ax.grid(alpha=0.3)
plt.show()

## 7 · Plot — choropleth, latest year

In [ ]:
import plotly.express as px

latest = df.year.max()
sub = df[df.year == latest]

fig = px.choropleth(
    sub, geojson=gj, locations="pc11_d_id",
    featureidkey="properties.pc11_d_id",
    color="volume_m3", color_continuous_scale="OrRd",
    hover_name="d_name",
    labels={"volume_m3": "Built-up volume (m³)"},
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(title=f"Built-up volume by district — {latest}",
                   margin=dict(l=0, r=0, t=40, b=0))
fig.show()

## 8 · Plot — top 20 districts, latest year

In [ ]:
top = sub.nlargest(20, "volume_m3").sort_values("volume_m3")
fig = px.bar(top, x="volume_m3", y="d_name", orientation="h",
             hover_data=["pc11_s_id"],
             labels={"volume_m3": "Built-up volume (m³)", "d_name": "District"},
             color_discrete_sequence=["#f57d6a"])
fig.update_layout(title=f"Top 20 districts by built-up volume — {latest}",
                  height=560, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 9 · Download the result

In [ ]:
try:
    from google.colab import files
    files.download(OUT_CSV)
except Exception:
    print("Not on Colab — file is at", OUT_CSV)

> **Caveat — the 2022 snapshot.** This series is the *raw* Open Buildings 2.5D
> output and has not been cleaned. In 2022 the model reports **10% more
> buildings** (the largest count rise in the series, in 97% of districts) while
> mean height falls **16.7%** and volume falls **19%** — it detected more
> buildings and simultaneously decided they were all much shorter. That is not
> a physical event. The timing coincides with Sentinel-2 Processing Baseline
> 04.00 (operational 2022-01-25), which added a band-dependent radiometric
> offset (`BOA_ADD_OFFSET`, typically -1000); mixing baseline eras without
> harmonising produces exactly this kind of permanent step. The shift does not
> revert in 2023.
>
> Practical consequence: `footprint_m2` and `building_count` are usable across
> the break, `volume_m3` and `mean_height_m` are not. Restrict height/volume
> work to 2016–2021, or carry a post-2021 dummy.
>
> Separately, note that `mean_height_m` runs high (~8.5 m nationally) because
> presence-weighting shrinks the denominator where model confidence is low.
> Trust the district *ordering*, treat the absolute level as an upper bound.